# Stage 1 — Non-Instruction Fine-Tuning

Domain adaptation of `Qwen/Qwen2.5-0.5B` on raw California homeowners-insurance text using [Unsloth](https://github.com/unslothai/unsloth), LoRA/QLoRA, on a free Colab T4.

Spec: `specs/002-unsloth-ca-homeowners-finetune.md` §6-7 (Stage 1).

**Run this notebook on Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). It has not been executed locally — there is no CUDA GPU in the local dev environment for this repo.

Before running: upload/clone this repo into the Colab session so that `data/1-non_instruction_data.txt` and the `src/` package are available at the paths used below (e.g. `!git clone <repo-url>` then `%cd <repo-name>`).

## 0. Install dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

In [ ]:
import sys
from pathlib import Path

# Assumes this notebook is run from notebooks/ inside the cloned repo, so the
# repo root (one level up) contains the src/ package and data/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "1-non_instruction_data.txt"
ADAPTER_OUT_DIR = REPO_ROOT / "models" / "stage1-non-instruction-adapter"
print("Repo root:", REPO_ROOT)
print("Data path exists:", DATA_PATH.exists())

## 1. Load raw domain text

In [ ]:
from src.data_utils import load_raw_paragraphs

paragraphs = load_raw_paragraphs(DATA_PATH)
print(f"Loaded {len(paragraphs)} paragraphs")
print(paragraphs[0][:300])

## 2. Load base model via Unsloth

In [ ]:
from src.model_utils import BASE_MODEL_NAME, MAX_SEQ_LENGTH, load_base_model

model, tokenizer = load_base_model(
    model_name=BASE_MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Loaded:", BASE_MODEL_NAME, "| eos_token:", tokenizer.eos_token)

## 3. Clean and chunk text into a training dataset

Paragraphs are already clean, standalone prose (see `data/1-non_instruction_data.txt` header). Each paragraph gets the tokenizer's EOS token appended so the model learns paragraph boundaries; `packing=True` below then concatenates/chunks these into `MAX_SEQ_LENGTH`-sized training blocks.

In [ ]:
from src.data_utils import build_text_dataset

dataset = build_text_dataset(paragraphs, eos_token=tokenizer.eos_token)
dataset

## 4. Apply LoRA / QLoRA adapters

In [ ]:
from src.model_utils import add_lora_adapters

model = add_lora_adapters(model)  # uses DEFAULT_LORA_CONFIG: r=16, alpha=16, dropout=0.05

## 5. Train on raw domain text

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=str(REPO_ROOT / "outputs" / "stage1"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # effective batch size 8, per spec §6.3
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=True,
    args=training_args,
)

train_result = trainer.train()
print(train_result.metrics)

## 6. Save the adapter

Colab local disk is not persistent across sessions — copy `ADAPTER_OUT_DIR` to Google Drive or push it to a private Hugging Face model repo after this cell (see spec §6.4).

In [ ]:
ADAPTER_OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUT_DIR))
print("Saved adapter to:", ADAPTER_OUT_DIR)

In [ ]:
# Optional: persist to Google Drive so the adapter survives session restarts.
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage1-non-instruction-adapter", dirs_exist_ok=True)

## 7. Test the model after non-instruction fine-tuning

These are raw continuation prompts (not questions/instructions — that's Stage 2), meant to check whether the model has picked up domain vocabulary, tone, and background knowledge from the raw text.

In [ ]:
from unsloth import FastLanguageModel

from src.generation_utils import generate

FastLanguageModel.for_inference(model)

test_prompts = [
    "Dwelling coverage, sometimes labeled Coverage A,",
    "The California FAIR Plan is",
    "The difference between actual cash value and replacement cost value is",
    "Earthquake coverage in California",
    "After a covered loss, homeowners are typically expected to",
]

for prompt in test_prompts:
    completion = generate(model, tokenizer, prompt, max_new_tokens=80, temperature=0.7)
    print(f"PROMPT: {prompt}\nCOMPLETION: {completion}\n{'-' * 80}")